In [1]:
%pip install "qiskit==2.2.3" "qiskit-machine-learning==0.9.0" "qiskit-optimization==0.7.0" "qiskit-algorithms==0.4.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 20.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 54.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.4 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=ec5f674bcf84d68d61b01cd3379caf1ddb984603e90ce9b06558131a8193dc9d
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [2]:
import sys
import importlib.metadata as metadata
import numpy as np
import scipy
import sklearn
import qiskit
import qiskit_machine_learning
import qiskit_optimization
import qiskit_algorithms

expected = {
    "qiskit": "2.2.3",
    "qiskit-machine-learning": "0.9.0",
    "qiskit-optimization": "0.7.0",
    "qiskit-algorithms": "0.4.0",
}
print("Python:", sys.version.split()[0])
for package in [*expected, "numpy", "scipy", "scikit-learn"]:
    installed = metadata.version(package)
    print(f"{package}: {installed}")
    if package in expected:
        assert installed == expected[package], (
            f"{package} 버전을 확인하고 설치 셀 실행 후 세션을 다시 시작하세요."
        )
print("패키지 import와 버전 확인 완료")

Python: 3.13.18
qiskit: 2.2.3
qiskit-machine-learning: 0.9.0
qiskit-optimization: 0.7.0
qiskit-algorithms: 0.4.0
numpy: 2.4.6
scipy: 1.16.3
scikit-learn: 1.6.1
패키지 import와 버전 확인 완료


In [3]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.algorithms.classifiers import QSVC

# ---------------------------------------------------------
# 1. 데이터 생성
# ---------------------------------------------------------
normal = np.array([[180, 320, 0, 0, 4], [250, 410, 0, 0, 6], [320, 520, 1, 0, 8], [410, 630, 0, 0, 10], [510, 720, 1, 0, 12], [620, 860, 0, 0, 14], [750, 980, 1, 0, 16], [900, 1200, 0, 0, 18], [1050, 1450, 1, 0, 20], [1200, 1600, 0, 0, 22], [1400, 1800, 1, 0, 24], [1600, 2100, 0, 0, 26]], dtype=float)
attack = np.array([[900, 1200, 2, 0, 20], [1200, 1600, 3, 0, 24], [1500, 2200, 2, 1, 28], [2100, 3000, 4, 0, 32], [2800, 3800, 3, 1, 36], [3400, 4600, 5, 0, 40], [4100, 5500, 4, 1, 44], [4800, 6300, 6, 0, 48], [5500, 7100, 5, 1, 52], [6200, 7900, 7, 0, 56], [6900, 8700, 6, 1, 60], [7600, 9500, 8, 1, 64]], dtype=float)
X = np.vstack([normal, attack])
y = np.array([0] * len(normal) + [1] * len(attack))

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=11, stratify=y
)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()


# ---------------------------------------------------------
# 3. 고전적 SVM
# ---------------------------------------------------------
print("고전적 SVM 학습 중...")
svm = SVC(kernel="rbf")
svm.fit(X_train, y_train)
pred_svm = svm.predict(X_test)
acc_svm = accuracy_score(y_test, pred_svm)
print("고전적 SVM 완료")

# ---------------------------------------------------------
# 4. Quantum SVM을 위한 양자 커널 계산
# ---------------------------------------------------------
print("\nQuantum SVM 학습 시작... ")

feature_map = zz_feature_map(feature_dimension=5, reps=2)
quantum_kernel = FidelityStatevectorKernel(feature_map=feature_map)

qsvc = QSVC(quantum_kernel=quantum_kernel)
qsvc.fit(X_train, y_train)
pred_qsvc = qsvc.predict(X_test)
acc_qsvc = accuracy_score(y_test, pred_qsvc)

# ---------------------------------------------------------
# 6. 최종 결과 출력
# ---------------------------------------------------------
print("\n=== 최종 결과 ===")
print("고전적 SVM 정확도:", acc_svm)
print("Quantum SVM 정확도:", acc_qsvc)

from sklearn.metrics import classification_report
print("학습/시험 크기:", X_train.shape, X_test.shape)
print("QSVM 분류 결과:")
print(classification_report(y_test, pred_qsvc, digits=3, zero_division=0))
K = quantum_kernel.evaluate(X_train)
assert np.allclose(K, K.T) and np.allclose(np.diag(K), 1.0)
assert np.linalg.eigvalsh(K).min() > -1e-8

고전적 SVM 학습 중...
고전적 SVM 완료

Quantum SVM 학습 시작... 

=== 최종 결과 ===
고전적 SVM 정확도: 0.875
Quantum SVM 정확도: 0.75
학습/시험 크기: (16, 5) (8, 5)
QSVM 분류 결과:
              precision    recall  f1-score   support

           0      0.750     0.750     0.750         4
           1      0.750     0.750     0.750         4

    accuracy                          0.750         8
   macro avg      0.750     0.750     0.750         8
weighted avg      0.750     0.750     0.750         8



입력 열의 순서는 송신 바이트 수(src_bytes), 수신 바이트 수(dst_bytes), 로그인 실패 횟수(failed_logins), root 권한 획득 여부(root_shell), 관찰 구간 내 연결 횟수(count)입니다. 바이트 수는 byte, 횟수는 회, 권한 획득 여부는 0 또는 1입니다. 고정 데이터 24건의 범위는 각각 180~7,600, 320~9,500, 0~8, 0~1, 4~64입니다. 정상과 공격 데이터는 각각 12건이며 라벨은 정상 0, 공격 1입니다.

처리 순서는 원자료 분할, 학습 자료로 정규화 기준 계산, 학습·시험 자료 변환, RBF SVM 학습, 양자 커널 QSVM 학습, 시험 평가입니다. 학습 자료는 16건, 시험 자료는 8건입니다. MinMaxScaler는 학습 자료에서만 최솟값과 최댓값을 구합니다. 시험 자료는 동일한 기준으로 변환하며 학습 범위를 벗어난 값은 0~1 밖으로 나올 수 있습니다.

zz_feature_map은 5개 특징을 5큐비트에 인코딩합니다. FidelityStatevectorKernel은 상태벡터의 내적 절댓값을 제곱하여 유사도를 계산합니다. 같은 입력끼리의 커널 값은 1이고, 서로 다른 입력의 유사도는 0~1입니다. QSVC는 이 커널 행렬을 사용하여 분류 경계를 학습합니다. 코드는 커널의 대칭성, 대각 원소와 양의 준정부호 성질을 확인합니다.

고정 데이터의 실행 예에서 RBF SVM 정확도는 0.875, QSVM 정확도는 0.750입니다. 정확도는 시험 8건 중 각각 7건과 6건을 맞혔다는 뜻입니다. 공격 클래스의 precision, recall, f1-score를 함께 확인합니다. 이 작은 교육용 데이터의 결과는 해당 분할에서의 성능을 나타내며, 산업 데이터의 일반화 성능과 양자 계산의 속도 우위는 별도 실험으로 평가합니다.

###